<a target="_blank" href="https://colab.research.google.com/github/autoharness/CarTool-Instruct/blob/main/fine_tuning/deployment/litert_lm/Export_to_LiteRT_LM.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Export model for LiteRT-LM deployment

This notebook demonstrates the process of exporting a fine-tuned model to the LiteRT-LM format.

## Environment Setup

In [ ]:
%pip uninstall -y torch torchao tensorflow
%pip install litert-torch==0.8.0 torch==2.9.1 torchao==0.15.0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

In [ ]:
import os

from litert_torch.generative.examples.gemma3 import gemma3
from litert_torch.generative.examples.qwen import qwen3
from litert_torch.generative.layers import kv_cache
from litert_torch.generative.utilities import converter, loader as loading_utils
from litert_torch.generative.utilities.export_config import ExportConfig

In [ ]:
TARGET_MODEL = "functiongemma"  # "functiongemma", "qwen3"
CHECKPOINT_DIR = "/content/drive/MyDrive/checkpoint" # change to the real model checkpoint directory
METADATA_PATH = f"/content/llm_metadata_{TARGET_MODEL}.textproto" # upload the metadata textproto file to Colab
LITERTLM_OUTPUT_DIR = "/content/output"

os.makedirs(LITERTLM_OUTPUT_DIR, exist_ok=True)

export_config = ExportConfig()
export_config.kvcache_layout = kv_cache.KV_LAYOUT_TRANSPOSED
export_config.mask_as_input = True

converter_kwargs = {
    "output_path": LITERTLM_OUTPUT_DIR,
    "output_name_prefix": TARGET_MODEL,
    "output_format": "litertlm",
    "prefill_seq_len": 2048,
    "kv_cache_max_len": 12288,
    # https://github.com/google-ai-edge/litert-torch/tree/main/litert_torch/generative/quantize
    "quantize": "dynamic_int8",
    "base_llm_metadata_path": METADATA_PATH,
    "export_config": export_config,
}

## Conversion

In [ ]:
def qwen_custom_loader(checkpoint_path):
    """Handles the missing lm_head"""
    state_dict = loading_utils.load_safetensors(checkpoint_path)
    if "lm_head.weight" not in state_dict:
        print("Notice: lm_head.weight not found, tying with embedding.")
        state_dict["lm_head.weight"] = state_dict["model.embed_tokens.weight"]
    return state_dict

if TARGET_MODEL == "functiongemma":
    # NOTE: DO NOT use the chat template from model repo, LiteRT-LM has built-in template for FunctionGemma.
    raw_model = gemma3.build_model_270m(CHECKPOINT_DIR)
    converter_kwargs["tokenizer_model_path"] = os.path.join(CHECKPOINT_DIR, "tokenizer.model")

elif TARGET_MODEL == "qwen3":
    raw_model = qwen3.build_4b_model(CHECKPOINT_DIR, custom_loader=qwen_custom_loader)
    converter_kwargs["hf_tokenizer_model_path"] = os.path.join(CHECKPOINT_DIR, "tokenizer.json")

    jinja_path = os.path.join(CHECKPOINT_DIR, "chat_template.jinja")
    with open(jinja_path, "r", encoding="utf-8") as f:
        converter_kwargs["jinja_prompt_template"] = f.read()

else:
    raise ValueError(f"Unknown TARGET_MODEL: '{TARGET_MODEL}'.")

print(f"Starting LiteRT-LM conversion for {TARGET_MODEL}...")
converter.convert_to_litert(raw_model, **converter_kwargs)
print(f"✅ Conversion completed successfully! Output saved to: {LITERTLM_OUTPUT_DIR}")